In [210]:
import pandas as pd
import numpy as np
import seaborn as sns
from collections import Counter
from sklearnex import patch_sklearn
patch_sklearn()
import os
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier,RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV,GridSearchCV
from sklearn.model_selection import train_test_split

from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, f1_score,classification_report


Intel(R) Extension for Scikit-learn* enabled (https://github.com/intel/scikit-learn-intelex)


In [20]:
base_dir = '../'
fibseg_dir = os.path.join(base_dir, 'fiber_segmentation_processed_data')


In [96]:
collagen_tb = pd.read_csv(os.path.join(fibseg_dir,'fiber_object_table.csv'))
collagen_tb.dropna(axis =0, inplace=True)
collagen_tb = collagen_tb[np.isfinite(collagen_tb.alignment_score)]#drop non finite elements
collagen_tb['Leap_ID'] = collagen_tb.fov.str.split('_',n = 1).str[0].str.upper()
biosamples_path = '/home/giuseppe/devices/Delta_Tissue/IMC/IMC_data/ExtraDocs/processed_response_RCB.csv'
biosamples =pd.read_csv(biosamples_path)
collagen_tb = collagen_tb.merge(biosamples,left_on='Leap_ID',right_on= 'LEAP_ID').drop(['LEAP_ID'],axis = 1)#add metadata on patient
collagen_tb = collagen_tb[collagen_tb['SAMPLE_TYPE_(CORE/RESECTION)']=='CORE']

In [237]:
X = collagen_tb[['major_axis_length',
       'minor_axis_length', 'orientation', 'area', 'eccentricity',
       'euler_number', 'alignment_score','Leap_ID']].groupby(by = 'Leap_ID').mean()
y = collagen_tb[['Leap_ID','new_Response']].drop_duplicates().set_index('Leap_ID')
y['new_Response'] = y.new_Response.replace('enR','nR')
y = X.merge(y,left_index=True,right_index=True)['new_Response']

In [238]:

le = LabelEncoder()
y = le.fit_transform(y)
le.inverse_transform([0,1])

array(['nR', 'pCR'], dtype=object)

In [239]:
X_train, X_test, y_train, y_test = train_test_split(   X, y, test_size=0.33)

In [240]:
param_distributions1 = {
   # 'preprocessor__hashenc__n_features':[4,10,20,100,1000],
    'max_features': [1, 5,10,20,50 ,100,None],
    "max_leaf_nodes": [10, 100, 1000, None],
    "min_samples_leaf": [1, 2, 5, 10, 20, 50, 100],
}
clf = RandomForestClassifier(max_features= 100,class_weight="balanced")
#regr.fit(X_train, y_train)
search_cv1 = RandomizedSearchCV(
    clf,
    param_distributions=param_distributions1,
    scoring="roc_auc",
    n_iter=100,
    random_state=0,
    n_jobs=-1,
)
search_cv1.fit(X_train, y_train)



/home/giuseppe/anaconda3/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/giuseppe/anaconda3/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/giuseppe/anaconda3/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/giuseppe/anaconda3/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/giuseppe/anaconda3/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/gius

RandomizedSearchCV(estimator=RandomForestClassifier(class_weight='balanced',
                                                    max_features=100),
                   n_iter=100, n_jobs=-1,
                   param_distributions={'max_features': [1, 5, 10, 20, 50, 100,
                                                         None],
                                        'max_leaf_nodes': [10, 100, 1000, None],
                                        'min_samples_leaf': [1, 2, 5, 10, 20,
                                                             50, 100]},
                   random_state=0, scoring='roc_auc')

In [241]:
def format_cv_results(param_distributions,search_cv):
    columns = [f"param_{name}" for name in param_distributions.keys()]
    columns += ["mean_test_score", "std_test_score"]
    cv_results = pd.DataFrame(search_cv.cv_results_)
    #cv_results["mean_test_error"] = -cv_results["mean_test_score"]
    #cv_results["std_test_error"] = cv_results["std_test_score"]
    
    return cv_results[columns].sort_values(by="mean_test_score").iloc[:-10:-1],cv_results
cv_results_filt,cv_results1 = format_cv_results(param_distributions1,search_cv1)
cv_results_filt

,param_max_features,param_max_leaf_nodes,param_min_samples_leaf,mean_test_score,std_test_score
67,20,None,10,0.732500,0.233726
36,None,100,10,0.732500,0.233726
88,10,1000,10,0.728333,0.236150
52,5,10,5,0.721667,0.294184
65,None,100,5,0.721667,0.294184
10,1,100,1,0.721667,0.294184
22,50,None,10,0.720000,0.237440
24,10,100,1,0.719167,0.248992
30,1,None,2,0.718333,0.266375


In [242]:
y_pred = search_cv1.predict(X_test)
aucroc = roc_auc_score(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
f1_ = f1_score(y_test, y_pred)
bal_acc_ = balanced_accuracy_score(y_test, y_pred)
mode = 'Test'
pcriterion = 'majority'
metrics = {
        f"{mode} Accuracy {pcriterion}": accuracy,
        f"{mode} AUC {pcriterion}": aucroc,
        f"{mode} F1 Score {pcriterion}": f1_,
        f"{mode} Balanced Accuracy": bal_acc_,
}
pd.Series(metrics)

/home/giuseppe/anaconda3/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


Test Accuracy majority    0.500000
Test AUC majority         0.484848
Test F1 Score majority    0.375000
Test Balanced Accuracy    0.484848
dtype: float64

In [233]:
print(classification_report(y_train, search_cv1.predict(X_train)))

              precision    recall  f1-score   support

           0       0.85      0.89      0.87        19
           1       0.89      0.84      0.86        19

    accuracy                           0.87        38
   macro avg       0.87      0.87      0.87        38
weighted avg       0.87      0.87      0.87        38



/home/giuseppe/anaconda3/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [195]:
from sklearn.metrics import f1_score

In [234]:
print(classification_report(y_test, search_cv1.predict(X_test)))

              precision    recall  f1-score   support

           0       0.67      0.46      0.55        13
           1       0.36      0.57      0.44         7

    accuracy                           0.50        20
   macro avg       0.52      0.52      0.49        20
weighted avg       0.56      0.50      0.51        20



/home/giuseppe/anaconda3/lib/python3.9/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [144]:
RandomForestClassifier?

Init signature:
RandomForestClassifier(
    n_estimators=100,
    *,
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features='sqrt',
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    bootstrap=True,
    oob_score=False,
    n_jobs=None,
    random_state=None,
    verbose=0,
    warm_start=False,
    class_weight=None,
    ccp_alpha=0.0,
    max_samples=None,
    monotonic_cst=None,
    max_bins=256,
    min_bin_size=1,
)
Docstring:     
A random forest classifier.

A random forest is a meta estimator that fits a number of decision tree
classifiers on various sub-samples of the dataset and uses averaging to
improve the predictive accuracy and control over-fitting.
Trees in the forest use the best split strategy, i.e. equivalent to passing
`splitter="best"` to the underlying :class:`~sklearn.tree.DecisionTreeRegressor`.
The sub-sample size is controlled with the `max_samples` parameter if
`boo